In [1]:
import json
import os
import shutil

def coco_to_yolo_cbdet(json_path, img_src_dir, img_dst_dir, label_dst_dir):
    """
    All 4 CBDet categories → class 1 (bag)
    No person class in this dataset
    """
    os.makedirs(img_dst_dir,    exist_ok=True)
    os.makedirs(label_dst_dir,  exist_ok=True)

    with open(json_path) as f:
        coco = json.load(f)

    # All categories → bag (class 1)
    class_map = {1: 1, 2: 1, 3: 1, 4: 1}

    images      = {img["id"]: img for img in coco["images"]}
    annotations = {}
    for ann in coco["annotations"]:
        img_id = ann["image_id"]
        if img_id not in annotations:
            annotations[img_id] = []
        annotations[img_id].append(ann)

    converted, skipped = 0, 0

    for img_id, img_info in images.items():
        fname = img_info["file_name"]
        img_w = img_info["width"]
        img_h = img_info["height"]

        src_img = os.path.join(img_src_dir, fname)
        if not os.path.exists(src_img):
            skipped += 1
            continue

        # Copy image
        shutil.copy(src_img, os.path.join(img_dst_dir, os.path.basename(fname)))

        # Write label
        label_fname = os.path.splitext(os.path.basename(fname))[0] + ".txt"
        label_path  = os.path.join(label_dst_dir, label_fname)

        lines = []
        for ann in annotations.get(img_id, []):
            cat_id   = ann["category_id"]
            yolo_cls = class_map.get(cat_id)
            if yolo_cls is None:
                continue

            x, y, w, h = ann["bbox"]  # COCO: top-left x, y, width, height
            cx = (x + w/2) / img_w
            cy = (y + h/2) / img_h
            bw = w / img_w
            bh = h / img_h

            # Clamp to valid range
            cx = max(0, min(1, cx))
            cy = max(0, min(1, cy))
            bw = max(0, min(1, bw))
            bh = max(0, min(1, bh))

            if bw > 0 and bh > 0:
                lines.append(f"{yolo_cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

        with open(label_path, "w") as f:
            f.writelines(lines)

        converted += 1

    print(f"  Converted: {converted} | Skipped: {skipped}")


# ── Run conversion ────────────────────────────────────────────────────────────

base_ann = "../../data/raw_data/bag6k/annotations"
base_img = "../../data/raw_data/bag6k/bag6k"
base_dst = "cbdet_yolo"

for split in ["train", "val", "test"]:
    json_path = os.path.join(base_ann, f"instances_{split}.json")
    if not os.path.exists(json_path):
        print(f"Skipping {split} — not found")
        continue

    print(f"\nConverting {split}...")
    coco_to_yolo_cbdet(
        json_path     = json_path,
        img_src_dir   = base_img,
        img_dst_dir   = f"{base_dst}/images/{split}",
        label_dst_dir = f"{base_dst}/labels/{split}",
    )

print("\nDone. CBDet YOLO format at ./cbdet_yolo")


Converting train...
  Converted: 3906 | Skipped: 0

Converting val...
  Converted: 647 | Skipped: 0

Converting test...
  Converted: 1952 | Skipped: 0

Done. CBDet YOLO format at ./cbdet_yolo


In [2]:
import os

cbdet_abs = os.path.abspath("cbdet_yolo")

yaml_content = f"""path: {cbdet_abs}
train: images/train
val:   images/val
test:  images/test

nc: 1

names:
  0: bag
"""

with open("cbdet_yolo/dataset.yaml", "w") as f:
    f.write(yaml_content)

print("Created cbdet_yolo/dataset.yaml")

Created cbdet_yolo/dataset.yaml


In [3]:
import os
from collections import Counter

for split in ["train", "val", "test"]:
    label_dir = f"cbdet_yolo/labels/{split}"
    if not os.path.exists(label_dir):
        continue

    counter   = Counter()
    n_files   = 0
    empty     = 0

    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue
        n_files += 1
        with open(os.path.join(label_dir, fname)) as f:
            lines = [l for l in f.readlines() if l.strip()]
        if not lines:
            empty += 1
        for line in lines:
            cls = int(line.split()[0])
            counter[cls] += 1

    print(f"{split}: {n_files} images | {counter[0]} bags | {empty} empty labels")

train: 3906 images | 0 bags | 10 empty labels
val: 647 images | 0 bags | 1 empty labels
test: 1952 images | 0 bags | 4 empty labels


In [4]:
import os
from collections import Counter

for split in ["train", "val", "test"]:
    label_dir = f"cbdet_yolo/labels/{split}"
    if not os.path.exists(label_dir):
        continue

    counter = Counter()
    n_files = 0
    empty   = 0

    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue
        n_files += 1
        with open(os.path.join(label_dir, fname)) as f:
            lines = [l for l in f.readlines() if l.strip()]
        if not lines:
            empty += 1
        for line in lines:
            cls = int(line.split()[0])
            counter[cls] += 1

    print(f"{split}: {n_files} images | {dict(counter)} | {empty} empty labels")

train: 3906 images | {1: 12875} | 10 empty labels
val: 647 images | {1: 2137} | 1 empty labels
test: 1952 images | {1: 6420} | 4 empty labels


In [5]:
import os

cbdet_abs = os.path.abspath("cbdet_yolo")

yaml_content = f"""path: {cbdet_abs}
train: images/train
val:   images/val
test:  images/test

nc: 2

names:
  0: person
  1: bag
"""

with open("cbdet_yolo/dataset.yaml", "w") as f:
    f.write(yaml_content)

print("Updated cbdet_yolo/dataset.yaml")
print(f"Path: {cbdet_abs}")


Updated cbdet_yolo/dataset.yaml
Path: /Users/anbu/Sri/CDS_2026Spring_project/notebook/exploratory/cbdet_yolo


In [1]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="cbdet_yolo/dataset.yaml",
    epochs=20,
    imgsz=640,
    batch=10,
    device="mps",
    project="yolov8n_cbdet",
    plots=True,
    cache=True,
    classes=[1],   # only train on bag — ignore the dummy person class
)

New https://pypi.org/project/ultralytics/8.4.26 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.21 🚀 Python-3.13.7 torch-2.10.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=10, bgr=0.0, box=7.5, cache=True, cfg=None, classes=[1], close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=cbdet_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, opti

In [ ]:
from ultralytics import YOLO
from collections import defaultdict
import numpy as np
import os
from PIL import Image

# CBDet trained bag model
bag_model  = YOLO("../../runs/detect/yolov8n_cbdet/train/weights/best.pt")
# COCO for people
coco_model = YOLO("yolov8n.pt")

val_img_dir   = "../../data/raw_data/final_unified_dataset/images/val"
val_label_dir = "../../data/raw_data/final_unified_dataset/labels/val"

# ── IoU ───────────────────────────────────────────────────────────────────────

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0]);  y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]);  y2 = min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

# ── Ground truth loader ───────────────────────────────────────────────────────

def load_ground_truth(label_path, img_w, img_h):
    gt_boxes, gt_labels = [], []
    if not os.path.exists(label_path):
        return gt_boxes, gt_labels
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls, cx, cy, bw, bh = map(float, parts)
            x1 = (cx - bw/2) * img_w
            y1 = (cy - bh/2) * img_h
            x2 = (cx + bw/2) * img_w
            y2 = (cy + bh/2) * img_h
            gt_boxes.append([x1, y1, x2, y2])
            gt_labels.append(int(cls))
    return gt_boxes, gt_labels

# ── AP computation ────────────────────────────────────────────────────────────

def compute_ap(detections, n_gt):
    if n_gt == 0:
        return 0.0
    detections.sort(key=lambda x: -x[0])
    tp_cumsum  = np.cumsum([d[1] for d in detections])
    precisions = tp_cumsum / (np.arange(len(detections)) + 1)
    recalls    = tp_cumsum / n_gt
    ap = 0
    for t in np.arange(0, 1.1, 0.1):
        prec = [p for p, r in zip(precisions, recalls) if r >= t]
        ap  += max(prec) if prec else 0
    return ap / 11

# ── Evaluation ────────────────────────────────────────────────────────────────

all_detections = defaultdict(list)
all_gt_counts  = defaultdict(int)

img_files = [f for f in os.listdir(val_img_dir) if f.endswith((".jpg", ".png"))]
print(f"Evaluating on {len(img_files)} val images...")

for i, fname in enumerate(img_files):
    if i % 50 == 0:
        print(f"  {i}/{len(img_files)}")

    img_path   = os.path.join(val_img_dir, fname)
    label_path = os.path.join(val_label_dir,
                    fname.replace(".jpg", ".txt").replace(".png", ".txt"))

    img      = Image.open(img_path)
    img_w, img_h = img.size

    gt_boxes, gt_labels = load_ground_truth(label_path, img_w, img_h)
    for lbl in gt_labels:
        all_gt_counts[lbl] += 1

    # CBDet model — bags only (class 1)
    bag_results  = bag_model.predict(img_path,  classes=[1], conf=0.29, verbose=False)
    # COCO — people only (class 0)
    coco_results = coco_model.predict(img_path, classes=[0], conf=0.40, verbose=False)

    predictions = []

    for box in bag_results[0].boxes:
        predictions.append((
            box.xyxy[0].cpu().tolist(),
            1,                           # bag
            float(box.conf[0])
        ))

    for box in coco_results[0].boxes:
        predictions.append((
            box.xyxy[0].cpu().tolist(),
            0,                           # person
            float(box.conf[0])
        ))

    # Match predictions to ground truth
    matched_gt = set()
    for pred_box, pred_cls, pred_conf in sorted(predictions, key=lambda x: -x[2]):
        gt_indices = [i for i, l in enumerate(gt_labels) if l == pred_cls]
        best_iou, best_i = 0, -1

        for gi in gt_indices:
            iou = compute_iou(pred_box, gt_boxes[gi])
            if iou > best_iou:
                best_iou, best_i = iou, gi

        is_tp = best_iou >= 0.5 and best_i not in matched_gt
        if is_tp:
            matched_gt.add(best_i)

        all_detections[pred_cls].append((pred_conf, is_tp))

# ── Results ───────────────────────────────────────────────────────────────────

class_names = {0: "person", 1: "bag"}
aps = {}

print("\n" + "="*55)
print("CBDET BAG MODEL + COCO — EVALUATED ON OPEN IMAGES VAL")
print("="*55)

for cls_id in sorted(all_detections.keys()):
    dets   = all_detections[cls_id]
    n_gt   = all_gt_counts[cls_id]
    ap     = compute_ap(dets, n_gt)
    aps[cls_id] = ap

    dets_sorted = sorted(dets, key=lambda x: -x[0])
    tp_cumsum   = np.cumsum([d[1] for d in dets_sorted])
    n_dets      = len(dets_sorted)
    precision   = tp_cumsum[-1] / n_dets if n_dets > 0 else 0
    recall      = tp_cumsum[-1] / n_gt   if n_gt   > 0 else 0

    source = "CBDet bag model" if cls_id == 1 else "COCO pretrained"
    print(f"\n{class_names[cls_id].upper()} (from {source})")
    print(f"  AP50:      {ap:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  GT count:  {n_gt}")
    print(f"  Detections:{n_dets}")

map50 = np.mean(list(aps.values()))
print(f"\nOverall mAP50: {map50:.4f}")
print("="*55)

# ── Full comparison table ─────────────────────────────────────────────────────

print("\nFull model comparison:")
print(f"{'Metric':<22} {'v8n single':>12} {'OI+COCO':>12} {'CBDet+COCO':>12}")
print("-" * 60)
print(f"{'mAP50':<22} {'0.556':>12} {'0.4417':>12} {map50:>12.4f}")
print(f"{'Person mAP50':<22} {'0.248':>12} {'0.1597':>12} {aps.get(0,0):>12.4f}")
print(f"{'Bag mAP50':<22} {'0.865':>12} {'0.7238':>12} {aps.get(1,0):>12.4f}")

Evaluating on 1171 val images...
  0/1171
  50/1171
